## For a given layer
* How many resources required?
* How long is the latency?

In [1]:
# test layer
test_layer = {"nix": 224, "niy": 224, "nif": 64, "nof": 64, "kernel": 3, "type": "conv"}

## Resource Cost
* DSP
    * $n^2 \times P_n \times P_m$
    * $n^2$: panel size of IFM; $P_m$: parallel input channel; $P_n$: parallel output channel
* BRAM
    * BRAM18K: width = 18-bits; depth = 1024
        * For example, buffer size = $32(bits) \times 512$, concatenating two units of BRAM18K to have wider datawidth
    * Weight buffer: $r^2 \times T_n \times T_m$.
    * IFM buffer: $W\times T_m\times n$  
    * IFM Line Buffer Temp: tile overlap area, $W\times T_m\times m$
    * OFM buffer: $2\times C \times T_n \times m$
    * Banks: 一個 Cycle 可存取的 data 量
        * Weight: $r^2 \times P_n \times P_m$
        * IFM: $n^2 \times P_m$
        * IFM Temp: $n\times m \times P_m$
        * OFM: $2\times m^2 \times P_n$, multiply by 2 since 32-bits accumulator
    * Depth: $\frac{\text{buffer size}}{\text{banks}}$
        * Weight: $\frac{r^2 \times T_n \times T_m}{r^2 \times P_n \times P_m} = \frac{T_n}{p_n}\times\frac{T_m}{p_m}$
        * IFM: $\frac{W\times T_m\times n}{n^2 \times P_m} = \frac{W}{n} \times \frac{T_m}{P_m}$
        * IFM Temp: $\frac{W\times T_m\times m}{n\times m \times P_m} = \frac{W}{n}\times\frac{T_m}{P_m}$
        * OFM: $\frac{2\times C \times T_n \times m}{2\times m^2 \times P_n} = \frac{C}{m}\times\frac{T_n}{P_n}$
        
    * Total BRAM units: $\frac{\text{datawidth}}{18} \times \frac{\text{depth}}{1024}$, $\text{datawidth} = \text{banks}\times 16(\text{bits})$
        * Weight: $\frac{r^2 \times P_n \times P_m\times 16}{18} \times \frac{\frac{T_n}{p_n}\times\frac{T_m}{p_m}}{1024}$
        * IFM: $\frac{n^2 \times P_m \times 16}{18} \times \frac{\frac{W}{n} \times \frac{T_m}{P_m}}{1024}$
        * IFM Temp: $\frac{n\times m \times P_m \times 16}{18} \times \frac{\frac{W}{n}\times\frac{T_m}{P_m}}{1024}$
        * OFM: $\frac{m^2 \times P_n \times 32}{18} \times \frac{\frac{C}{m}\times\frac{T_n}{P_n}}{1024}$
        
    

In [2]:
from math import ceil
def dsp_cost(n, Pn, Pm):
    return (n**2) * Pn * Pm

def bram_cost(W, C, r, n, m, Tn, Tm, Pn, Pm, precision = 16, double_buffering = True):
    weight = ceil(((r**2) * Pn * Pm * precision)/18) * ceil((ceil(Tn/Pn)*ceil(Tm/Pm)) / 1024)
    IFM = ceil(((n**2) * Pm * precision)/18) * ceil((ceil(W/n) * ceil(Tm/Pm))/1024)
    IFM_temp = ceil((n*m*Pm*precision) / 18) * ceil((ceil(W/n)*ceil(Tm/Pm))/1024)
    OFM = ceil(((m**2)*Pn*32)/18) * ceil((ceil(C/m)*ceil(Tn/Pn))/1024)
    
    print(weight, IFM, IFM_temp, OFM)
    factor = 1
    if double_buffering:
        factor = 2
    
    return factor*(weight + IFM + IFM_temp + OFM)

In [3]:
W = 224; C=224; r=3; n=8; m=6; Tn=64; Tm=64; Pn=2; Pm=2; precision = 16
bram_cost(W, C, r, n, m, Tn, Tm, Pn, Pm, precision)

32 114 86 256


976

## Modeling Performance
* Tile comutation latency for one tile
    * $T_{\text{compute}} = (\frac{C}{m} \times \frac{T_m}{P_m} \times \frac{T_n}{P_n} \times \text{II} + P_{\text{depth}}) \times \frac{1}{\text{Freq}}$
    * $\text{II} = 1$

* Data transfer time: for input and output data
    * $T_{\text{transfer}} = \frac{n \times W \times \max(T_n, T_m) \times 16}{\text{datawidth}}$

* Initial time: loading IFM and WGT
    * $T_{\text{init}} = \frac{(T_m \times T_n \times r^2 + n \times W \times T_m) \times 16}{\text{datawidth}}$
    
* Total Latency
    * $T_{\text{total}} = \frac{M}{T_m} \times \frac{N}{T_n} \times (\frac{H}{m} \times T_{\text{compute}} + T_{\text{init}})$

In [37]:
def latency_tile(W, C, n, m, Tn, Tm, Pn, Pm, datawidth, II=1, precision=16, freq=0.2):
    compute_time = (ceil(C/m) * ceil(Tm / Pm) * ceil(Tn/Pn) * II) * (1/freq) * (10**(-6))
    transfer_time = ceil((n*W*max(Tn,Tm)*precision)/datawidth) * (1/freq) * (10**(-6))
    print("compute_time > transfer_time:", compute_time > transfer_time)
    return max(compute_time, transfer_time)

def latency_init(W, r, n , Tn, Tm, datawidth, precision=16, freq = 16):
    init_time = ceil((Tm*Tn*r*r + n*W*Tm)*precision / datawidth) * (1/freq) * (10**(-6))
    return init_time

def latency_total(W, C, M, N, H, r, n, m, Tn, Tm, Pn, Pm, datawidth, II=1, precision = 16, freq = 0.2):
    total_time = ceil(M/Tm) * ceil(N/Tn) * (ceil(H/m) * \
                        latency_tile(W, C, n, m, Tn, Tm, Pn, Pm, datawidth, II, precision, freq) + \
                        latency_init(W, r, n , Tn, Tm, datawidth, precision))
    return total_time;

In [43]:
W = 224; H=224; C=112; M = 20; N = 64; r=7; n=8; m=6; Tn=64; Tm=64; Pn=2; Pm=2; II=1; datawidth=512; precision = 16
latency_total(W, C, M, N, H, r, n, m, Tn, Tm, Pn, Pm, datawidth)

compute_time > transfer_time: True


3.6972559999999994